# 230 · e-f Rabi on a shared drive port

**Level:** medium · **Topics:** Gen2, SHFQC, dsl, sections, timing, calibration, sweeps · **Type:** write the code

An e-f Rabi experiment drives two transitions of the same qubit:

1. a g-e π pulse brings the qubit to `e`,
2. an e-f pulse follows immediately, its **amplitude is swept**,
3. the readout follows immediately.

The qubit has one drive line, so both pulses leave from the **same output** of the SHFQC+, at two different frequencies.

Cells with a `TODO` are yours to complete. Everything runs in emulation, no instrument needed.

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.simple import *

## Device setup

Two logical signals, `drive` and `drive_ef`, are connected to the same port.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/drive_ef", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Step 1: calibration

The g-e transition is at 4.10 GHz. The e-f transition is at **3.93 GHz**.

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Calibration properties](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/02_calibration_properties.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)
# TODO: calibrate the e-f drive at 3.93 GHz, with hardware modulation
calibration["q0/drive_ef"] = SignalCalibration(
    range=10,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=80e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=-25,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=80e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=10,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Step 2: the sequence

For every amplitude: g-e pulse on `drive`, then e-f pulse on `drive_ef` with the swept amplitude, then the readout.

📖 [Section tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/00_section_tutorial.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html)

In [ ]:
ge_pulse = pulse_library.gaussian(length=23e-9, amplitude=0.7)
ef_pulse = pulse_library.gaussian(length=16e-9, amplitude=1.0)
readout_pulse = pulse_library.const(length=2e-6, amplitude=0.5)

amplitudes = LinearSweepParameter(start=0.1, stop=1.0, count=10)

In [ ]:
@dsl.experiment(signals=["drive", "drive_ef", "measure", "acquire"])
def ef_rabi():
    with dsl.acquire_loop_rt(count=10):
        # TODO: sweep `amplitudes`
        # TODO: a section that plays `ge_pulse` on "drive"
        # TODO: a section that plays `ef_pulse` on "drive_ef" with the swept
        #       amplitude, after the g-e pulse
        # TODO: make sure the readout starts after the e-f pulse
        with dsl.section(name="readout"):
            dsl.measure(
                measure_signal="measure",
                acquire_signal="acquire",
                measure_pulse=readout_pulse,
                integration_kernel=readout_pulse,
                handle="result",
                reset_delay=1e-6,
            )


experiment = ef_rabi()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "drive_ef": "q0/drive_ef",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Step 3: compile and run

Once steps 1 and 2 are complete, the compilation most likely fails with `Overlapping HW oscillators`.
Read the message and fix the sequence. Keep the pulse lengths and keep hardware modulation.

📖 [Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)

**Expected result:** `(10,)`

Look at the first shot. Both pulses are on the same output: the g-e pulse ends at 24 ns, the e-f pulse runs from 24 ns to 40 ns, and the readout starts at 40 ns.

In [ ]:
plot_simulation(compiled, start_time=0, length=150e-9, signals=["drive", "measure"])

## Questions

1. Why can the e-f pulse not start at 23 ns, right at the end of the g-e pulse?
2. With a g-e pulse of 24 ns the first version would have compiled. Why?
3. Can `drive` and `drive_ef` have different local oscillator frequencies?
4. Would software modulation avoid the problem?

<details>
<summary>Hint 1</summary>

Both signals use the same port, so they share its local oscillator. Only the oscillator frequency (IF) is different: 3.93 GHz − 4.0 GHz.
</details>

<details>
<summary>Hint 2</summary>

The error message gives two ways out: "add a delay, or ensure the 2nd pulse starts on the system grid".
`dsl.section` has an `on_system_grid` argument.
</details>

<details>
<summary>Hint 3</summary>

If a small gap appears between the two pulses, look at the `alignment` of the first section.
</details>

The solution is in [solution.ipynb](solution.ipynb).